# Rock energy segmentation

Финальная версия первого notebook.

Цель:

1. взять очищенную телеметрию бурения;
2. посчитать proxy энергоёмкости через `pseudo_mse`;
3. получить непрерывный индекс `hardness_score_smooth`;
4. разделить участки на 4 класса энергоёмкости;
5. сохранить размеченный CSV для advisory-модели и симулятора;
6. построить Plotly surfaces для классов энергоёмкости.

В этой версии `hardness_score` считается только через `log_pseudo_mse`.
`formation_residual`, `expected_speed_from_controls` и `drilling_efficiency` не используются для разметки.

In [3]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

import plotly.graph_objects as go

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
EPS = 1e-9

INPUT_PATH = Path("../datasets/united.csv")
OUTPUT_PATH = Path("united_rock_energy_segment_quantile.csv")

ARTIFACT_DIR = Path("rock_energy_segment_quantile_artifacts")
SURFACE_HTML_DIR = Path("plotly_surfaces_html")
REPORT_DIR = Path("rock_energy_segment_reports")

ARTIFACT_DIR.mkdir(exist_ok=True)
SURFACE_HTML_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

SEGMENT_SIZE = 60
ROLL_WINDOWS = [12, 30, 60]

ENERGY_LABELS_4 = [
    "soft_low_energy",
    "medium_low_energy",
    "medium_high_energy",
    "hard_high_energy",
]

print("Input:", INPUT_PATH.resolve())
print("Output:", OUTPUT_PATH.resolve())

Input: C:\Users\qa1ro\OneDrive\Рабочий стол\diploma\OptimalDrilling\datasets\united.csv
Output: C:\Users\qa1ro\OneDrive\Рабочий стол\diploma\OptimalDrilling\notebooks\united_rock_energy_segment_quantile.csv


## 1. Загрузка данных

Notebook ожидает очищенный датасет. Очистка здесь не выполняется.

In [4]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {INPUT_PATH}. Put cleaned source dataset near this notebook or change INPUT_PATH."
    )

df = pd.read_csv(INPUT_PATH)
print("Loaded:", df.shape)
display(df.head())

required = ["well_id", "pressure_axis", "pressure_rotation", "rotation", "speed"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if "depth_m" not in df.columns and "depth" in df.columns:
    df["depth_m"] = df["depth"]

if "processing_time" in df.columns:
    df["processing_time"] = pd.to_datetime(df["processing_time"], errors="coerce")
    sort_cols = ["well_id", "processing_time"]
elif "depth_m" in df.columns:
    sort_cols = ["well_id", "depth_m"]
else:
    sort_cols = ["well_id"]

df = df.sort_values(sort_cols).reset_index(drop=True)
print("Sorted by:", sort_cols)

Loaded: (415049, 8)


,Unnamed: 0,processing_time,depth,rotation,pressure_axis,pressure_rotation,well_id,speed
0,0,2025-10-07 22:18:14.495,4.6965,104.892,12781,13144,25512,0.02020
1,1,2025-10-07 22:18:19.239,4.8783,101.430,18971,18936,25512,0.03636
2,2,2025-10-07 22:18:24.208,5.0298,101.430,20153,18184,25512,0.03030
3,3,2025-10-07 22:18:29.385,5.1813,102.714,20162,17445,25512,0.03030
4,4,2025-10-07 22:19:22.224,6.0297,103.110,18591,19741,25512,0.02424


Sorted by: ['well_id', 'processing_time']


## 2. Базовые энергетические признаки

Главная логика:

```text
energy_input_proxy = pressure_axis + pressure_rotation * rotation
pseudo_mse = energy_input_proxy / speed
```

`pseudo_mse` показывает, сколько proxy-энергии приходится на единицу скорости проходки.
Чем выше `pseudo_mse`, тем выше энергоёмкость/сопротивление участка.

In [5]:
df["total_pressure"] = df["pressure_axis"] + df["pressure_rotation"]
df["pressure_balance"] = df["pressure_axis"] / (df["total_pressure"] + EPS)
df["rotation_efficiency"] = df["rotation"] / (df["pressure_rotation"] + EPS)
df["axis_x_rotation"] = df["pressure_axis"] * df["rotation"]

df["energy_input_proxy"] = df["pressure_axis"] + df["pressure_rotation"] * df["rotation"]
df["pseudo_mse"] = df["energy_input_proxy"] / (df["speed"].clip(lower=EPS) + EPS)
df["log_pseudo_mse"] = np.log1p(df["pseudo_mse"].clip(lower=0))

# Диагностический показатель. В hardness_score не используется.
df["drilling_efficiency"] = df["speed"] / (df["energy_input_proxy"] + EPS)

display(df[[
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "speed",
    "energy_input_proxy",
    "pseudo_mse",
    "log_pseudo_mse",
    "drilling_efficiency",
]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)

,count,mean,std,min,1%,5%,50%,95%,99%,max
pressure_axis,415049.0,1.747367e+04,4.682983e+03,3.170000e+02,3.713000e+03,6.645000e+03,1.886100e+04,2.234300e+04,2.362600e+04,2.487200e+04
pressure_rotation,415049.0,1.413415e+04,3.243523e+03,7.840000e+02,6.271000e+03,8.246000e+03,1.463700e+04,1.875860e+04,2.088100e+04,2.631800e+04
rotation,415049.0,1.039453e+02,1.337036e+01,5.001000e+01,6.498000e+01,8.175000e+01,1.031580e+02,1.384740e+02,1.390200e+02,1.395780e+02
speed,415049.0,1.311610e-02,6.615073e-03,1.001653e-03,2.754545e-03,5.050000e-03,1.212000e-02,2.424000e-02,3.030000e-02,3.895714e-02
energy_input_proxy,415049.0,1.494500e+06,4.009753e+05,4.174649e+04,4.697380e+05,7.635049e+05,1.542361e+06,2.119089e+06,2.462635e+06,3.668551e+06
pseudo_mse,415049.0,1.467940e+08,9.159134e+07,3.009424e+06,3.257211e+07,5.222076e+07,1.240000e+08,3.036946e+08,4.643063e+08,2.583497e+09
log_pseudo_mse,415049.0,1.864997e+01,5.525713e-01,1.491726e+01,1.729897e+01,1.777099e+01,1.863579e+01,1.953153e+01,1.995606e+01,2.167241e+01
drilling_efficiency,415049.0,9.279041e-09,5.931875e-09,3.870720e-10,2.153750e-09,3.292781e-09,8.064516e-09,1.914947e-08,3.070111e-08,3.322895e-07


## 3. Rolling features

Используем скользящие медианы, чтобы индекс энергоёмкости отражал локальный участок, а не одиночный шумный замер.

Для rolling-окон используется неполное начальное окно:

```text
min_periods = max(3, window // 3)
```

Это важно: мы не выкидываем первые 59 строк каждой скважины для окна 60, как было бы при `min_periods=60`. Такая логика соответствует предыдущей успешной версии pipeline и сохраняет больше данных для advisory-моделей.


In [6]:
ROLL_COLS = [
    "energy_input_proxy",
    "pseudo_mse",
    "log_pseudo_mse",
    "drilling_efficiency",
    "rotation",
    "speed",
    "pressure_axis",
    "pressure_rotation",
]

for col in ROLL_COLS:
    grp = df.groupby("well_id")[col]
    for w in ROLL_WINDOWS:
        min_p = max(3, w // 3)
        df[f"{col}_roll_median_{w}"] = grp.transform(lambda s: s.rolling(w, min_periods=min_p).median())
        df[f"{col}_roll_mean_{w}"] = grp.transform(lambda s: s.rolling(w, min_periods=min_p).mean())
        df[f"{col}_roll_std_{w}"] = grp.transform(lambda s: s.rolling(w, min_periods=min_p).std())

print("Added rolling features.")

Added rolling features.


## 4. Hardness score по `log_pseudo_mse`

Финальная формула:

```text
hardness_score = zscore(log_pseudo_mse)
hardness_score_smooth = zscore(log1p(pseudo_mse_roll_median_60))
```

`hardness_score_smooth` используется для сегментации, потому что он устойчивее к одиночным скачкам скорости и давления.

In [7]:
def zscore(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    return (s - s.mean()) / (s.std(ddof=0) + EPS)

df["hardness_score"] = zscore(df["log_pseudo_mse"])

smooth_source = np.log1p(df["pseudo_mse_roll_median_60"].clip(lower=0))
df["hardness_score_smooth"] = zscore(smooth_source)

display(df[["hardness_score", "hardness_score_smooth"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)

,count,mean,std,min,1%,5%,50%,95%,99%,max
hardness_score,415049.0,8.063965e-16,1.000001,-6.755168,-2.444936,-1.590703,-0.025652,1.595392,2.363659,5.469786
hardness_score_smooth,382635.0,-2.681171e-15,1.000001,-6.033240,-2.697221,-1.737822,0.089445,1.660352,2.115974,3.724367


## 5. Сегментация и 4 класса энергоёмкости

Сначала строки внутри каждой скважины делятся на последовательные сегменты.
Затем для каждого сегмента считается медианный `hardness_score_smooth`.
После этого сегменты делятся на 4 квантиля.

In [8]:
df["row_in_well"] = df.groupby("well_id").cumcount()
df["segment_idx_in_well"] = (df["row_in_well"] // SEGMENT_SIZE).astype(int)
df["segment_id"] = df["well_id"].astype(str) + "_" + df["segment_idx_in_well"].astype(str)

segment_df = (
    df.dropna(subset=["hardness_score_smooth"])
    .groupby(["well_id", "segment_id"], as_index=False)
    .agg(
        segment_start_row=("row_in_well", "min"),
        segment_end_row=("row_in_well", "max"),
        n_rows=("hardness_score_smooth", "size"),
        hardness_segment=("hardness_score_smooth", "median"),
        pseudo_mse_segment=("pseudo_mse_roll_median_60", "median"),
        speed_segment=("speed", "median"),
        drilling_efficiency_segment=("drilling_efficiency", "median"),
    )
)

segment_df["energy_type_segment_quantile"] = pd.qcut(
    segment_df["hardness_segment"],
    q=4,
    labels=ENERGY_LABELS_4,
    duplicates="drop",
)

if segment_df["energy_type_segment_quantile"].isna().any():
    raise ValueError("qcut produced NaN labels. Check hardness distribution.")

segment_df["energy_type_segment_quantile"] = segment_df["energy_type_segment_quantile"].astype(str)

df = df.merge(
    segment_df[["segment_id", "hardness_segment", "energy_type_segment_quantile"]],
    on="segment_id",
    how="left",
)

df["rock_energy_type_final"] = df["energy_type_segment_quantile"]

print("Segments:", segment_df.shape)
display(segment_df.head())
display(segment_df["energy_type_segment_quantile"].value_counts().sort_index())

Segments: (7763, 10)


,well_id,segment_id,segment_start_row,segment_end_row,n_rows,hardness_segment,pseudo_mse_segment,speed_segment,drilling_efficiency_segment,energy_type_segment_quantile
0,19601,19601_0,19,59,41,-1.041624,8.002831e+07,0.01515,1.230014e-08,soft_low_energy
1,19601,19601_1,60,119,60,0.239057,1.326085e+08,0.00606,4.991257e-09,medium_high_energy
2,19601,19601_2,120,179,60,1.328915,2.038022e+08,0.00606,4.965333e-09,hard_high_energy
3,19601,19601_3,180,239,60,0.231340,1.322048e+08,0.01212,7.682533e-09,medium_high_energy
4,19601,19601_4,240,299,60,0.373971,1.398526e+08,0.01212,7.262727e-09,medium_high_energy


energy_type_segment_quantile
hard_high_energy      1941
medium_high_energy    1940
medium_low_energy     1941
soft_low_energy       1941
Name: count, dtype: int64

## 6. Summary таблицы

Эти CSV сохраняются для будущего отдельного analysis notebook.

In [9]:
summary_by_energy = (
    df.dropna(subset=["rock_energy_type_final"])
    .groupby("rock_energy_type_final", as_index=False)
    .agg(
        n_rows=("speed", "size"),
        n_wells=("well_id", "nunique"),
        speed_mean=("speed", "mean"),
        speed_median=("speed", "median"),
        speed_p10=("speed", lambda s: s.quantile(0.10)),
        speed_p90=("speed", lambda s: s.quantile(0.90)),
        pseudo_mse_median=("pseudo_mse", "median"),
        pseudo_mse_smooth_median=("pseudo_mse_roll_median_60", "median"),
        drilling_efficiency_median=("drilling_efficiency", "median"),
        drilling_efficiency_smooth_median=("drilling_efficiency_roll_median_60", "median"),
        hardness_score_smooth_median=("hardness_score_smooth", "median"),
        energy_input_proxy_median=("energy_input_proxy", "median"),
        pressure_axis_median=("pressure_axis", "median"),
        pressure_rotation_median=("pressure_rotation", "median"),
        rotation_median=("rotation", "median"),
    )
)

segment_summary = (
    segment_df
    .groupby("energy_type_segment_quantile", as_index=False)
    .agg(
        n_segments=("segment_id", "size"),
        n_wells=("well_id", "nunique"),
        hardness_segment_median=("hardness_segment", "median"),
        pseudo_mse_segment_median=("pseudo_mse_segment", "median"),
        speed_segment_median=("speed_segment", "median"),
        drilling_efficiency_segment_median=("drilling_efficiency_segment", "median"),
    )
)

display(summary_by_energy)
display(segment_summary)

summary_by_energy.to_csv(REPORT_DIR / "energy_type_summary.csv", index=False)
segment_summary.to_csv(REPORT_DIR / "segment_energy_type_summary.csv", index=False)
segment_df.to_csv(REPORT_DIR / "segments_with_energy_type.csv", index=False)

print("Saved reports to:", REPORT_DIR)

,rock_energy_type_final,n_rows,n_wells,speed_mean,speed_median,speed_p10,speed_p90,pseudo_mse_median,pseudo_mse_smooth_median,drilling_efficiency_median,drilling_efficiency_smooth_median,hardness_score_smooth_median,energy_input_proxy_median,pressure_axis_median,pressure_rotation_median,rotation_median
0,hard_high_energy,104327,690,0.009158,0.00606,0.00505,0.01515,1.859193e+08,1.843800e+08,5.378676e-09,5.432527e-09,1.074936,1530992.530,19564.0,14423.0,103.458
1,medium_high_energy,102336,915,0.012091,0.01212,0.00606,0.01818,1.364405e+08,1.352311e+08,7.329203e-09,7.395542e-09,0.288753,1622780.334,20283.0,15466.0,102.864
2,medium_low_energy,103015,1010,0.014047,0.01212,0.00606,0.02020,1.121490e+08,1.080458e+08,8.916707e-09,9.256987e-09,-0.280392,1586327.740,19043.0,15091.0,103.008
3,soft_low_energy,105371,981,0.017121,0.01818,0.00606,0.02525,7.924527e+07,7.684046e+07,1.261905e-08,1.301561e-08,-1.144709,1325885.418,15697.0,12638.0,103.458


,energy_type_segment_quantile,n_segments,n_wells,hardness_segment_median,pseudo_mse_segment_median,speed_segment_median,drilling_efficiency_segment_median
0,hard_high_energy,1941,690,1.058675,1.832062e+08,0.006818,5.487151e-09
1,medium_high_energy,1940,915,0.285795,1.350735e+08,0.012120,7.328479e-09
2,medium_low_energy,1941,1010,-0.287974,1.077233e+08,0.013635,8.794310e-09
3,soft_low_energy,1941,981,-1.152796,7.659599e+07,0.018180,1.200450e-08


Saved reports to: rock_energy_segment_reports


## 7. Plotly surfaces для классов энергоёмкости

Surface-модели нужны только для визуализации в симуляторе.
Они строят приближённую карту:

```text
pressure_axis × pressure_rotation → speed
```

внутри каждого класса энергоёмкости.

In [10]:
SURFACE_FEATURES = [
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "hardness_score_smooth",
    "pressure_balance",
    "rotation_efficiency",
    "energy_input_proxy",
]

surface_reports = []

for surface_type in ENERGY_LABELS_4:
    surface_train = (
        df[df["rock_energy_type_final"] == surface_type]
        .dropna(subset=SURFACE_FEATURES + ["speed"])
        .copy()
    )

    if len(surface_train) < 500:
        print("Skip small class:", surface_type, len(surface_train))
        continue

    surface_model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=RANDOM_STATE,
        )),
    ])

    surface_model.fit(surface_train[SURFACE_FEATURES], surface_train["speed"])
    pred = surface_model.predict(surface_train[SURFACE_FEATURES])

    surface_reports.append({
        "rock_energy_type_final": surface_type,
        "n_rows": int(len(surface_train)),
        "mae": float(mean_absolute_error(surface_train["speed"], pred)),
        "rmse": float(root_mean_squared_error(surface_train["speed"], pred)),
        "r2": float(r2_score(surface_train["speed"], pred)),
        "pressure_axis_q05": float(surface_train["pressure_axis"].quantile(0.05)),
        "pressure_axis_q95": float(surface_train["pressure_axis"].quantile(0.95)),
        "pressure_rotation_q05": float(surface_train["pressure_rotation"].quantile(0.05)),
        "pressure_rotation_q95": float(surface_train["pressure_rotation"].quantile(0.95)),
    })

    fixed_state = surface_train[SURFACE_FEATURES].median(numeric_only=True)

    p_ax_grid = np.linspace(
        surface_train["pressure_axis"].quantile(0.05),
        surface_train["pressure_axis"].quantile(0.95),
        60,
    )
    p_rot_grid = np.linspace(
        surface_train["pressure_rotation"].quantile(0.05),
        surface_train["pressure_rotation"].quantile(0.95),
        60,
    )

    PA, PR = np.meshgrid(p_ax_grid, p_rot_grid)

    grid = pd.DataFrame({
        "pressure_axis": PA.ravel(),
        "pressure_rotation": PR.ravel(),
    })

    for col in SURFACE_FEATURES:
        if col not in grid.columns:
            grid[col] = fixed_state[col]

    grid["pressure_balance"] = grid["pressure_axis"] / (grid["pressure_axis"] + grid["pressure_rotation"] + EPS)
    grid["rotation_efficiency"] = grid["rotation"] / (grid["pressure_rotation"] + EPS)
    grid["energy_input_proxy"] = grid["pressure_axis"] + grid["pressure_rotation"] * grid["rotation"]

    Z = surface_model.predict(grid[SURFACE_FEATURES]).reshape(PA.shape)

    fig = go.Figure()
    fig.add_trace(go.Surface(
        x=PA,
        y=PR,
        z=Z,
        colorscale="Viridis",
        opacity=0.95,
        showscale=True,
    ))

    fig.update_layout(
        title=f"Surface by energy type: {surface_type}",
        scene=dict(
            xaxis_title="pressure_axis",
            yaxis_title="pressure_rotation",
            zaxis_title="speed",
        ),
        height=800,
    )

    surface_html = SURFACE_HTML_DIR / f"surface_{surface_type}.html"
    fig.write_html(surface_html, include_plotlyjs=True, full_html=True)
    print("Saved:", surface_html)

surface_report_df = pd.DataFrame(surface_reports)
display(surface_report_df)
surface_report_df.to_csv(REPORT_DIR / "surface_model_report.csv", index=False)

Saved: plotly_surfaces_html\surface_soft_low_energy.html
Saved: plotly_surfaces_html\surface_medium_low_energy.html
Saved: plotly_surfaces_html\surface_medium_high_energy.html
Saved: plotly_surfaces_html\surface_hard_high_energy.html


,rock_energy_type_final,n_rows,mae,rmse,r2,pressure_axis_q05,pressure_axis_q95,pressure_rotation_q05,pressure_rotation_q95
0,soft_low_energy,88651,0.004395,0.005614,0.387257,6881.00,21278.50,8091.0,18535.00
1,medium_low_energy,95510,0.003850,0.004908,0.309544,11280.15,22309.00,9464.0,19141.00
2,medium_high_energy,98156,0.003440,0.004410,0.246981,13666.75,22704.00,10364.5,18818.00
3,hard_high_energy,100318,0.003026,0.003912,0.304590,13078.00,22726.15,9642.0,18819.15


## 8. Сохранение итогового CSV и config

In [11]:
df.to_csv(OUTPUT_PATH, index=False)

config = {
    "method": "log_pseudo_mse_only_segment_quantile",
    "input_path": str(INPUT_PATH),
    "output_path": str(OUTPUT_PATH),
    "segment_size": SEGMENT_SIZE,
    "rolling_windows": ROLL_WINDOWS,
    "rolling_min_periods": "max(3, window // 3)",
    "energy_labels_4": ENERGY_LABELS_4,
    "hardness_score": "zscore(log_pseudo_mse)",
    "hardness_score_smooth": "zscore(log1p(pseudo_mse_roll_median_60))",
    "surface_features": SURFACE_FEATURES,
    "removed_from_hardness_score": [
        "expected_speed_from_controls",
        "formation_residual",
        "relative_formation_residual",
        "drilling_efficiency",
    ],
}

with open("rock_energy_segment_quantile_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

with open(ARTIFACT_DIR / "rock_energy_segment_quantile_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved labeled CSV:", OUTPUT_PATH.resolve())
print("Saved config: rock_energy_segment_quantile_config.json")
print("Saved config:", ARTIFACT_DIR / "rock_energy_segment_quantile_config.json")
print("Final shape:", df.shape)

Saved labeled CSV: C:\Users\qa1ro\OneDrive\Рабочий стол\diploma\OptimalDrilling\notebooks\united_rock_energy_segment_quantile.csv
Saved config: rock_energy_segment_quantile_config.json
Saved config: rock_energy_segment_quantile_artifacts\rock_energy_segment_quantile_config.json
Final shape: (415049, 97)
